# Lab 01: Vectorized NumPy, Copy-on-Write Pandas & Analytical SQL Window Functions

**Companion Module:** [Module 01: Python, SQL & Data Stack](../../curriculum/01-foundations/01_python_sql_and_data_stack.md)  
**Difficulty:** Core | **Environment:** Pure Python 3.11+, NumPy, Pandas, SQLite

---

## 1. Problem Statement
In production data engineering for machine learning, data pipelines suffer from three common architectural bottlenecks:
1. **Slow loops:** Applying non-vectorized Python loops over multi-million row numerical arrays.
2. **Data mutation bugs:** Silent `SettingWithCopyWarning` and unintentional parent DataFrame mutation.
3. **Complex feature calculation:** Writing convoluted application code to calculate rolling historical metrics instead of leveraging in-database analytical SQL window functions.

---

## 2. Learning Objectives
By the end of this lab, you will be able to:
- Standardize multi-feature arrays using NumPy multi-axis broadcasting without Python loops.
- Perform group-level median imputation safely under Pandas Copy-on-Write (CoW).
- Formulate analytical SQL window functions (`LAG`, rolling averages, `ROW_NUMBER`) in SQLite.

---

## 3. Required Dependencies
```bash
pip install numpy pandas
```


In [ ]:
import sqlite3
import numpy as np
import pandas as pd

# Enforce modern Pandas Copy-on-Write mode
pd.options.mode.copy_on_write = True
print(f"NumPy Version: {np.__version__} | Pandas Version: {pd.__version__}")


---

## 4. Dataset Description
We simulate a transactional feature dataset representing user activity logs:
- `user_id`: Customer identifier string.
- `tier`: Subscription tier (`'pro'` vs. `'free'`).
- `spend`: Monthly transaction volume in USD (containing realistic missing values).
- `ts`: Monotonically increasing transaction timestamp.


In [ ]:
# 5. Implementation Part 1: Vectorized Z-Score Standardization via NumPy Broadcasting
X = np.array([
    [10.0, 200.0],
    [14.0, 180.0],
    [8.0, 220.0],
    [16.0, 240.0]
], dtype=float)

# Compute feature-wise mean and standard deviation along axis 0
mean_vec = X.mean(axis=0)
std_vec = X.std(axis=0)

# Broadcasted transformation: (4, 2) - (2,) / (2,)
X_standardized = (X - mean_vec) / np.maximum(std_vec, 1e-8)

assert X_standardized.shape == (4, 2)
assert np.allclose(X_standardized.mean(axis=0), 0.0)
assert np.allclose(X_standardized.std(axis=0), 1.0)
print("Standardized Feature Matrix:\n", X_standardized)


In [ ]:
# 5. Implementation Part 2: Safe Group-Level Median Imputation (Pandas CoW)
raw_df = pd.DataFrame({
    'user_id': ['u1', 'u2', 'u3', 'u4', 'u5'],
    'tier': ['pro', 'pro', 'free', 'free', 'free'],
    'spend': [120.0, np.nan, 20.0, 35.0, np.nan],
})

# Create explicit missingness indicator feature
df = raw_df.copy()
df['spend_missing'] = df['spend'].isna().astype(int)

# Safe group median imputation via transform without chained indexing
df['spend'] = df['spend'].fillna(df.groupby('tier')['spend'].transform('median'))

assert df['spend'].isna().sum() == 0
assert df.loc[df['user_id'] == 'u2', 'spend'].iloc[0] == 120.0
assert df.loc[df['user_id'] == 'u5', 'spend'].iloc[0] == 27.5
print("Imputed DataFrame:\n", df)


In [ ]:
# 5. Implementation Part 3: Analytical SQL Window Functions in SQLite
conn = sqlite3.connect(':memory:')
conn.execute('CREATE TABLE tx (user_id TEXT, ts INT, amount REAL)')
conn.executemany('INSERT INTO tx VALUES (?, ?, ?)', [
    ('u1', 1, 50.0),
    ('u1', 2, 70.0),
    ('u1', 3, 120.0),
    ('u2', 1, 30.0),
    ('u2', 2, 90.0),
])

sql_query = """
SELECT
    user_id,
    ts,
    amount,
    -- Previous transaction amount via LAG
    LAG(amount, 1, 0.0) OVER (PARTITION BY user_id ORDER BY ts ASC) AS prev_amount,
    -- Rolling 2-transaction average
    AVG(amount) OVER (
        PARTITION BY user_id ORDER BY ts ASC
        ROWS BETWEEN 1 PRECEDING AND CURRENT ROW
    ) AS rolling_2_avg,
    -- Sequential transaction index per user
    ROW_NUMBER() OVER (PARTITION BY user_id ORDER BY ts ASC) AS tx_seq
FROM tx
ORDER BY user_id, ts ASC;
"""

tx_df = pd.read_sql_query(sql_query, conn)
assert tx_df.loc[2, 'prev_amount'] == 70.0
assert np.isclose(tx_df.loc[2, 'rolling_2_avg'], 95.0)
print("Analytical Window Output:\n", tx_df)


---

## 6. Results and Observations
1. **NumPy Broadcasting:** Subtracting `mean_vec` (shape `(2,)`) from `X` (shape `(4, 2)`) automatically expanded dimensions without manual loops, reducing runtime complexity from $O(N \cdot D)$ interpreter steps to compiled C-speed memory passes.
2. **Copy-on-Write Integrity:** Mutating `df['spend']` in Pandas 2.0+ left the original `raw_df` pristine without throwing `SettingWithCopyWarning`.
3. **Window Functions:** SQLite computes rolling metrics and lags within partitioned memory structures, eliminating the need to write complex iterative state trackers in Python.

---

## 7. Hands-on Exercises
- **Exercise 1:** Modify the SQL query to compute `LEAD(amount, 1, 0.0)` to detect future lookahead values.
- **Exercise 2:** Implement a generator function that yields sliding chunks of 2 rows from `X` with 1-row overlap.

---

## 8. Further Exploration
- Explore [DuckDB](https://duckdb.org/) for multi-threaded SQL execution directly on Parquet files without database imports.
- Review [Pandas 2.0+ Copy-on-Write Guide](https://pandas.pydata.org/docs/user_guide/copy_on_write.html).
